# 🛰️ TRISAT: Suite Completa EUSPA Galileo OSNMA v1.1, E6 HAS-UA (PPP) & OS SIS v2.2

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jreyes13/TRISAT/blob/main/notebooks/02_euspa_osnma_has_complete_suite.ipynb)

Este cuaderno interactivo documenta y ejecuta paso a paso la arquitectura completa de **TRISAT** y **TRISAT-SDR**, implementando los estándares oficiales de la **EUSPA (European Union Agency for the Space Programme)**:

1. **Arquitectura SDR Multi-Banda (Dual-HackRF Sincronizado):** Generación de códigos PRN y modulación `CBOC(6,1,1/11)` en **E1-B/C** ($1575.42\text{ MHz}$) + `BPSK(5)` en **E6-B/C HAS** ($1278.75\text{ MHz}$) + **E5a/E5b AltBOC**.
2. **Motor Criptográfico Galileo OSNMA v1.1 (Vectores Reales Anexo A):**
   - Validación del **Árbol de Merkle (`DSM-PKR`)** y padding `P_DP`.
   - Descompresión de clave pública en curva elíptica **NIST P-256 (`secp256r1`)** y verificación de firma **ECDSA-SHA256** sobre `DSM-KROOT`.
   - Cadena **TESLA One-Way ($K_{14} \to \dots \to K_4 \to K_0$)** con sal $\alpha = \text{0x610BDF26D77B}$.
   - Verificación de `MACSEQ = 0xB08` y tags `HMAC-SHA-256` truncados a 40 bits (`ADKD=0, 4, 12`).
3. **Algoritmo de Usuario de Alta Precisión Galileo HAS (`EUSPA HAS-UA v2.0`):**
   - Decodificación **Reed-Solomon `RS(255, 32, d=224)`** sobre $\text{GF}(2^8)$ ($p(x)=x^8+x^4+x^3+x^2+1$, `0x11D`).
   - Transformación orbital **NTW $\to$ ECEF** (Ec. 19), selección de **Satélite Pivote** (Ec. 80), filtro **Single-Difference PPP (`PPPSD`)** y control de calidad **DIA (Test Global LOM + Baarda $w$-test)**.
   - Modelo ionosférico **NTCM-G de 12 coeficientes** para usuarios monofrecuencia.
4. **Servicios Avanzados Galileo OS SIS ICD v2.2:**
   - `Word Type 16` (**RedCED** para TTFF rápido), `Word Types 17..20` (**Reed-Solomon `RS(118,58)` Outer FEC2**), `Word Type 22` (**ARAIM ISM**), `Word Type 44` (**Timing Service Message - TSM**) y geocerca elíptica del servicio de alerta de emergencia **`EWSS-CAMF` (122 bits)**.

## 0. Preparación del Entorno (Autocontenido para Google Colab y Local)

La siguiente celda clona automáticamente el repositorio `TRISAT` si estás ejecutando este cuaderno en **Google Colab**, o añade la raíz del proyecto al `sys.path` si lo ejecutas en local.

In [ ]:
import os
import sys
import math
import hashlib
import hmac
import numpy as np
import matplotlib.pyplot as plt

# Si estamos en Google Colab y no existe trisat_sdr, clonamos el repositorio
if not os.path.exists('trisat_sdr') and not os.path.exists('../trisat_sdr'):
    print('🌐 Entorno Google Colab detectado. Clonando repositorio TRISAT...')
    !git clone --depth 1 https://github.com/jreyes13/TRISAT.git /tmp/TRISAT_repo 2>/dev/null || true
    if os.path.exists('/tmp/TRISAT_repo'):
        sys.path.insert(0, '/tmp/TRISAT_repo')
else:
    repo_root = os.path.abspath('..') if os.path.exists('../trisat_sdr') else os.path.abspath('.')
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

plt.style.use('seaborn-v0_8-darkgrid' if 'seaborn-v0_8-darkgrid' in plt.style.available else 'default')
print('✅ Entorno TRISAT listo.')

---
## 1. Arquitectura RF Dual-HackRF y Modulación `CBOC(6,1,1/11)` vs `BPSK(5)`

En **TRISAT-SDR**, dos front-ends SDR (**HackRF One #0 Maestro** y **HackRF One #1 Esclavo** sincronizados mediante el reloj de referencia `CLKOUT -> CLKIN` de $10\text{ MHz}$) capturan simultáneamente:

| Front-End SDR | Banda GNSS | Frecuencia Central | Modulación | Tasa de Chip | Servicio EUSPA |
| :--- | :--- | :---: | :---: | :---: | :--- |
| **HackRF #0 (Master)** | **Galileo E1-B/C + GPS L1** | $1575.42\text{ MHz}$ | `CBOC(6,1,1/11)` | $1.023\text{ Mcps}$ | **I/NAV v2.2 + OSNMA (40 bits)** |
| **HackRF #1 (Slave)** | **Galileo E6-B/C** | $1278.75\text{ MHz}$ | `BPSK(5)` | $5.115\text{ Mcps}$ | **C/NAV HAS (`RS(255,32)` PPP)** |
| **HackRF #1 (Alt.)** | **Galileo E5a / E5b / AltBOC** | $1176.45 / 1191.795\text{ MHz}$ | `AltBOC(15,10)` | $10.23\text{ Mcps}$ | **F/NAV + I/NAV Iono-Free** |

La señal **Galileo E1-B** emplea la modulación compuesta $\text{CBOC}(6, 1, 1/11)$ en fase:
$$s_{\text{E1-B}}(t) = \sqrt{\frac{10}{11}}\,\text{sc}_{\text{BOC}(1,1)}(t) - \sqrt{\frac{1}{11}}\,\text{sc}_{\text{BOC}(6,1)}(t)$$

In [ ]:
# Visualización de la forma de onda subportadora Galileo E1 CBOC(6,1,1/11) y pico de autocorrelación
t_chip = np.linspace(0, 2, 1000)  # 2 chips de duración
boc11 = np.sign(np.sin(2 * np.pi * 1.0 * t_chip))
boc61 = np.sign(np.sin(2 * np.pi * 6.0 * t_chip))
w1 = np.sqrt(10.0 / 11.0)
w6 = np.sqrt(1.0 / 11.0)
cboc_e1b = w1 * boc11 - w6 * boc61
cboc_e1c = w1 * boc11 + w6 * boc61

# Función de autocorrelación teórica BPSK(1) vs BOC(1,1) vs CBOC(6,1,1/11) vs E6 BPSK(5)
tau = np.linspace(-1.2, 1.2, 1200)  # Retardo en chips de E1 (1 chip = 977.5 ns)
r_bpsk1 = np.maximum(0.0, 1.0 - np.abs(tau))
r_boc11 = np.piecewise(np.abs(tau), [np.abs(tau) <= 0.5, (np.abs(tau) > 0.5) & (np.abs(tau) <= 1.0)],
                       [lambda x: 1.0 - 3.0*x, lambda x: x - 1.0, 0.0])
r_bpsk5 = np.maximum(0.0, 1.0 - 5.0 * np.abs(tau))  # E6 BPSK(5): chip 5 veces más estrecho

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.2))
ax1.plot(t_chip, cboc_e1b, color='#06b6d4', lw=2, label='E1-B Data CBOC(6,1,1/11, -)')
ax1.plot(t_chip, boc11, color='#94a3b8', ls='--', alpha=0.7, label='BOC(1,1) Base')
ax1.set_title('Subportadora Galileo E1-B CBOC(6,1,1/11) (4 niveles)', fontweight='bold')
ax1.set_xlabel('Tiempo [Chips de E1, 1 chip = 977.5 ns]')
ax1.set_ylabel('Amplitud Normalizada')
ax1.legend(loc='upper right')

ax2.plot(tau, r_bpsk1, color='#64748b', ls=':', lw=1.8, label='GPS L1 C/A BPSK(1)')
ax2.plot(tau, r_boc11, color='#06b6d4', lw=2.2, label='Galileo E1 BOC/CBOC')
ax2.plot(tau, r_bpsk5, color='#a855f7', lw=2.2, label='Galileo E6 HAS BPSK(5)')
for tap_x, tap_lbl in [(-0.5, 'VE'), (-0.25, 'E'), (0.0, 'P'), (0.25, 'L'), (0.5, 'VL')]:
    ax2.axvline(tap_x, color='#f59e0b', ls='--', alpha=0.45)
    ax2.text(tap_x, 1.03, tap_lbl, ha='center', fontsize=9, fontweight='bold', color='#d97706')
ax2.set_title('Pico de Autocorrelación y 5 Taps del Correlador (VE, E, P, L, VL)', fontweight='bold')
ax2.set_xlabel('Error de Fase de Código τ [Chips E1]')
ax2.set_ylabel('Correlación Normalizada R(τ)')
ax2.set_ylim(-0.6, 1.15)
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 2. Motor Criptográfico Galileo OSNMA v1.1 (Vectores Oficiales EUSPA Anexo A)

Según los documentos **Galileo OSNMA SIS ICD v1.1** y **Galileo OSNMA Receiver Guidelines v1.3 (Anexo A)**, cada página impar de **E1-B I/NAV** transporta **40 bits de OSNMA** (`8 bits HKROOT` + `32 bits MACK`). Al cabo de un subframe de 30 segundos (15 páginas nominales) se reconstruyen:
- **Bloque `HKROOT` (120 bits):** Contiene el estado `NMAS`, `CID`, `CPKS` y un bloque del mensaje `DSM-PKR` (clave pública ECDSA autenticada contra la raíz del **Árbol de Merkle**) o `DSM-KROOT` (raíz de la cadena TESLA firmada con **ECDSA P-256**).
- **Bloque `MACK` (480 bits):** Contiene los tags truncados `HMAC-SHA-256` (`ADKD=0` efemérides, `ADKD=4` tiempo/ionosfera, `ADKD=12` Slow MAC), el campo de integridad de secuencia `MACSEQ` (12 bits) y la clave **TESLA** $K_I$ del subframe anterior.

A continuación ejecutamos la **verificación criptográfica completa del Anexo A de la EUSPA** (Satélite `E02`, `WN=1248`, `TOW=345660`):

In [ ]:
try:
    from trisat_sdr.nav.osnma_engine import GalileoOsnmaEngine, verify_ecdsa_p256_sha256
    osnma = GalileoOsnmaEngine()
except ImportError:
    osnma = None

# Vectores Oficiales EUSPA OSNMA Receiver Guidelines v1.3 - Anexo A (Configuración 2)
ANNEX_A_MERKLE_ROOT = '7332397B4E23D2DFB69580088190B8669D4AC2E8774E0A2B9D3B039784369E68'
ANNEX_A_LEAF_M1     = '040203522563F605C027C3792767E783135AA2D651961C6C19C7B76E912C3258A79D'
ANNEX_A_PDP         = 'B8884C0D33D6'
ANNEX_A_NPK         = '03522563F605C027C3792767E783135AA2D651961C6C19C7B76E912C3258A79D'
ANNEX_A_KROOT_M     = '04221B01610BDF26D77B4E02A35BF8C9CBFCF70422081475FD445DF0FF'
ANNEX_A_KROOT_DS    = (
    '32721996A850C38A8500229D37360D86690B25A132F6F99A51D536D11ADFF72C'
    '8DB9066510DAE21E08F4F5C0B0E7559D77F3798B4D4214C562135E7408D0DB98'
)
ANNEX_A_ALPHA       = '610BDF26D77B'
ANNEX_A_K4          = '69C00AA7364237A65EBF006AD8DDBC73'
ANNEX_A_K0_KROOT    = '5BF8C9CBFCF70422081475FD445DF0FF'

# 1. Verificar Padding P_DP del mensaje DSM-PKR
pdp_calc = hashlib.sha256(bytes.fromhex(ANNEX_A_MERKLE_ROOT + ANNEX_A_LEAF_M1)).digest()[:6].hex().upper()
print(f'[1] DSM-PKR Padding P_DP      : {pdp_calc} == {ANNEX_A_PDP} -> {"✅ OK" if pdp_calc == ANNEX_A_PDP else "❌ FAIL"}')

# 2. Verificar Cadena TESLA One-Way (K_4 -> K_3 -> K_2 -> K_1 -> K_0 = KROOT)
def tesla_step(k_next_hex: str, wn: int, tow: int, alpha_hex: str) -> str:
    gst32 = ((wn & 0xFFF) << 20) | (tow & 0xFFFFF)
    data = bytes.fromhex(k_next_hex) + gst32.to_bytes(4, 'big') + bytes.fromhex(alpha_hex)
    return hashlib.sha256(data).digest()[:16].hex().upper()

k3 = tesla_step(ANNEX_A_K4, 1248, 345660, ANNEX_A_ALPHA)
k2 = tesla_step(k3,         1248, 345630, ANNEX_A_ALPHA)
k1 = tesla_step(k2,         1248, 345600, ANNEX_A_ALPHA)
k0 = tesla_step(k1,         1248, 345570, ANNEX_A_ALPHA)
print(f'[2] Cadena TESLA K4 -> K3     : {k3}')
print(f'                 K3 -> K2     : {k2}')
print(f'                 K2 -> K1     : {k1}')
print(f'                 K1 -> KROOT  : {k0} -> {"✅ OK" if k0 == ANNEX_A_K0_KROOT else "❌ FAIL"}')

# 3. Verificar MACSEQ (12 bits) en WN=1248, TOW=345660, PRN_A=2
gst_sf = ((1248 & 0xFFF) << 20) | (345660 & 0xFFFFF)
macseq_msg = bytes([2]) + gst_sf.to_bytes(4, 'big') + (0x1205).to_bytes(2, 'big') + (0x050F).to_bytes(2, 'big')
macseq_digest = hmac.new(bytes.fromhex(ANNEX_A_K4), macseq_msg, hashlib.sha256).digest()
macseq_12b = (int.from_bytes(macseq_digest[:2], 'big') >> 4) & 0xFFF
print(f'[3] Verificación MACSEQ (12b) : 0x{macseq_12b:03X} == 0xB08 -> {"✅ OK" if macseq_12b == 0xB08 else "❌ FAIL"}')

# 4. Verificar Tag HMAC-SHA-256 (ADKD=4, 40 bits)
tag2_msg = bytes.fromhex('02024E05463C03BFFFFFFFC00000449238227897FDEFF93040')
tag2_hex = hmac.new(bytes.fromhex(ANNEX_A_K4), tag2_msg, hashlib.sha256).digest()[:5].hex().upper()
print(f'[4] Tag HMAC-SHA256 (ADKD=4)  : 0x{tag2_hex} == 0x7BB238C883 -> {"✅ OK" if tag2_hex == "7BB238C883" else "❌ FAIL"}')

if osnma is not None:
    ecdsa_ok = verify_ecdsa_p256_sha256(ANNEX_A_NPK, bytes.fromhex(ANNEX_A_KROOT_M), ANNEX_A_KROOT_DS)
    print(f'[5] Firma ECDSA P-256 KROOT   : {"✅ VERIFICADA (secp256r1)" if ecdsa_ok else "❌ FAIL"}')

---
## 3. Galileo E6 High Accuracy Service (`EUSPA HAS-UA v2.0`)

El documento **Galileo HAS Reference User Algorithm Technical Note (`HAS-UA`)** define el procesamiento completo desde las páginas **C/NAV de E6-B** hasta el filtro de Kalman extendido **Single-Difference PPP (`PPPSD`)**:

1. **Decodificación Reed-Solomon `RS(255, 32, d=224)` sobre $\text{GF}(2^8)$:**
   Cada página C/NAV proporciona 1 palabra de información HAS de 424 bits (53 octetos en $\text{GF}(2^8)$). Al acumular $k$ páginas distintas de un mismo `MID`, se invierte la submatriz de Vandermonde $D_{k \times k}$ sobre $\text{GF}(2^8)$:
   $$M = D^{-1} \cdot P$$
2. **Transformación de Correcciones de Órbita `NTW` $\to$ `ECEF` (Ec. 19):**
   Con los vectores unitarios $\vec{e}_A = \frac{\vec{v}_{\text{BC}}}{\|\vec{v}_{\text{BC}}\|}$, $\vec{e}_C = \frac{\vec{r}_{\text{BC}} \times \vec{v}_{\text{BC}}}{\|\vec{r}_{\text{BC}} \times \vec{v}_{\text{BC}}\|}$ y $\vec{e}_R = \vec{e}_A \times \vec{e}_C$:
   $$\vec{r}_{\text{HAS}} = \vec{r}_{\text{BC}} - \begin{bmatrix} \vec{e}_R & \vec{e}_A & \vec{e}_C \end{bmatrix} \begin{bmatrix} \Delta r \\ \Delta a \\ \Delta c \end{bmatrix}$$
3. **Control de Integridad `DIA` (Detección, Identificación y Adaptación):**
   - **Test Global `LOM` (Local Overall Model):** $T_{\text{LOM}} = \frac{\hat{v}^T Q_{yy}^{-1} \hat{v}}{m - n} \sim F_{1-\alpha}(m-n, \infty)$.
   - **Test de Baarda ($w$-test):** $|w_i| = \frac{|e_i^T Q_{yy}^{-1} \hat{v}|}{\sqrt{e_i^T Q_{yy}^{-1} Q_{\hat{v}\hat{v}} Q_{yy}^{-1} e_i}} \le N_{1-\alpha_B/2}(0, 1) \approx 3.29$.

In [ ]:
# Simulación de convergencia del filtro EKF PPPSD de Galileo E6 HAS (SL1 < 20 cm H / < 40 cm V)
epochs_sec = np.arange(0, 305, 5)
np.random.seed(42)

# Error Horizontal 95%: OS Broadcast (~1.45 m) vs E6 HAS PPPSD (converge a ~0.14 m en < 150 s)
err_h_os  = 1.35 + 0.18 * np.sin(epochs_sec / 25.0) + np.random.normal(0, 0.06, len(epochs_sec))
err_h_has = 0.13 + 0.85 * np.exp(-epochs_sec / 45.0) + np.random.normal(0, 0.012, len(epochs_sec))
err_v_has = 0.21 + 1.10 * np.exp(-epochs_sec / 50.0) + np.random.normal(0, 0.018, len(epochs_sec))

fig, ax = plt.subplots(figsize=(10, 4.5))
ax.plot(epochs_sec, err_h_os, color='#94a3b8', ls='--', lw=1.8, label='Galileo E1 OS Autónoma (Sin HAS)')
ax.plot(epochs_sec, err_v_has, color='#a855f7', lw=2.2, label='Galileo E6 HAS-UA PPPSD Error Vertical (3D)')
ax.plot(epochs_sec, err_h_has, color='#10b981', lw=2.6, label='Galileo E6 HAS-UA PPPSD Error Horizontal (2D)')
ax.axhline(0.20, color='#06b6d4', ls=':', lw=1.6, label='Objetivo EUSPA HAS SL1 Horizontal (20 cm, 95%)')
ax.axhline(0.40, color='#f59e0b', ls=':', lw=1.4, label='Objetivo EUSPA HAS SL1 Vertical (40 cm, 95%)')

ax.set_title('Convergencia del Algoritmo EUSPA HAS-UA (Filtro EKF Single-Difference PPPSD)', fontweight='bold')
ax.set_xlabel('Tiempo desde arranque en frío [segundos]')
ax.set_ylabel('Error de Posicionamiento [metros]')
ax.set_ylim(0, 1.8)
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## 4. Modelo Ionosférico `NTCM-G` (12 Coeficientes) & Alerta de Emergencia `EWSS-CAMF`

Además de OSNMA y HAS-UA, **TRISAT** incorpora:
1. **Modelo Ionosférico `NTCM-G` (*Alternate Ionospheric Correction Algorithm for Galileo SF Users*):** Calcula el contenido total de electrones verticales ($\text{VTEC}$) mediante 5 factores multiplicativos ($F_1$ hora local/diurna, $F_2$ estacional, $F_3$ latitud geomagnética/anomalía ecuatorial, $F_4$ cresta geomagnética y $F_5$ flujo solar efectivo $Az$) y aplica la función de mapeo oblicuo a $h_{\text{Iono}} = 450\text{ km}$.
2. **Servicio de Alerta de Emergencia por Satélite (`EWSS-CAMF`, 122 bits):** Decodifica el elipsoide de alerta sobre WGS84 (campos `A12..A16`: centro $(\phi_c, \lambda_c)$, semieje mayor $L_M$, semieje menor $L_m$ y acimut $\theta$) y evalúa en tiempo real si el receptor se encuentra dentro del área de emergencia.

In [ ]:
from trisat_sdr.nav.has_ua_engine import NtcmGIonosphereModel
from trisat_sdr.nav.galileo_advanced_services import GalileoAdvancedServicesEngine

# 1. Evaluar el retardo oblicuo NTCM-G en banda E1 (1575.42 MHz) en función de la elevación del satélite
elevations = np.linspace(5, 90, 50)
delays_m = [
    NtcmGIonosphereModel.compute_slant_delay_m(40.4168, -3.7038, float(el), 140.0)[0]
    for el in elevations
]

# 2. Visualizar el área elíptica de alerta EWSS-CAMF (122 bits) alrededor del receptor
adv = GalileoAdvancedServicesEngine()
rep = adv.evaluate(rx_lat=40.4168, rx_lon=-3.7038)
ew = rep.ewss_alert

theta_rad = math.radians(ew.azimuth_deg)
ang = np.linspace(0, 2 * np.pi, 200)
u_km = ew.semi_major_km * np.cos(ang)
v_km = ew.semi_minor_km * np.sin(ang)
dx_km = u_km * np.cos(theta_rad) - v_km * np.sin(theta_rad)
dy_km = u_km * np.sin(theta_rad) + v_km * np.cos(theta_rad)

lat_ellipse = ew.ellipse_center_lat + dy_km / 111.132
lon_ellipse = ew.ellipse_center_lon + dx_km / (111.132 * math.cos(math.radians(ew.ellipse_center_lat)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.4))
ax1.plot(elevations, delays_m, color='#06b6d4', lw=2.4)
ax1.fill_between(elevations, delays_m, color='#06b6d4', alpha=0.15)
ax1.set_title('Retardo Ionosférico Oblicuo NTCM-G en Banda E1 (Madrid)', fontweight='bold')
ax1.set_xlabel('Elevación del Satélite [grados]')
ax1.set_ylabel('Retardo Ionosférico E1 [metros]')

ax2.plot(lon_ellipse, lat_ellipse, color='#ef4444', lw=2.2, label=f'Elipse EWSS ({ew.semi_major_km}x{ew.semi_minor_km} km)')
ax2.fill(lon_ellipse, lat_ellipse, color='#ef4444', alpha=0.15)
ax2.scatter([ew.ellipse_center_lon], [ew.ellipse_center_lat], color='#f59e0b', s=70, label='Centro Alerta (A12/A13)')
ax2.scatter([-3.7038], [40.4168], color='#10b981', s=90, marker='^', label='Receptor TRISAT (Dentro: Sí)')
ax2.set_title(f'Geocerca Elíptica Galileo EWSS-CAMF ({ew.message_type})', fontweight='bold')
ax2.set_xlabel('Longitud WGS84 [grados]')
ax2.set_ylabel('Latitud WGS84 [grados]')
ax2.legend(loc='upper right')
plt.tight_layout()
plt.show()

print('📌 Resumen Servicios Galileo OS SIS v2.2:')
print(f'   • I/NAV WT16 RedCED : Activo (TTFF = {rep.wt16_ttff_sec} s)')
print(f'   • I/NAV WT17-20 RS  : {rep.rs_ced_words_collected}/8 palabras -> Reconstrucción CED OK')
print(f'   • Patrón SSP actual : {rep.current_ssp_id}')
print(f'   • WT22 ARAIM ISM    : {rep.wt22_araim_ism_status}')
print(f'   • WT44 Timing (TSM) : {rep.wt44_tsm_service_level}')
print(f'   • EWSS-CAMF (122b)  : 0x{ew.camf_122bit_hex}')

---
## 5. Ejecución de una Época Completa del Receptor `TrisatSdrReceiver` y Exportación `.sol` (33 Columnas)

Finalmente ejecutamos una época completa del motor unificado `TrisatSdrReceiver` y mostramos la línea de solución en el formato oficial de **33 columnas `.sol`** definido en el *Galileo HAS-UA Software User Manual*:

In [ ]:
from trisat_sdr.config import ReceiverConfig
from trisat_sdr.receiver import TrisatSdrReceiver

cfg = ReceiverConfig()
cfg.telemetry.udp_enabled = False
cfg.telemetry.tcp_enabled = False
cfg.telemetry.pty_serial_enabled = False

rx = TrisatSdrReceiver(cfg)
rx.initialize_acquisition()
snapshot = rx.step_epoch()

print('================================================================================')
print(' 🛰️  SNAPSHOT DE TELEMETRÍA UNIFICADA TRISAT-SDR (OSNMA + HAS-UA + SIS v2.2)')
print('================================================================================')
print(f" Modo PVT          : {snapshot['pvt']['solution_mode']}")
print(f" Coordenadas WGS84 : ({snapshot['pvt']['latitude_deg']:.6f}, {snapshot['pvt']['longitude_deg']:.6f}) ±{snapshot['pvt']['horizontal_accuracy_m']:.2f} m")
print(f" Satélites Pivote  : Galileo E{snapshot['has_ua']['galileo_pivot_svid']:02d} | GPS G{snapshot['has_ua']['gps_pivot_svid']:02d}")
print(f" Test DIA (LOM/w)  : LOM = {snapshot['has_ua']['lom_test_statistic']} | max|w| = {snapshot['has_ua']['max_baarda_w_test']}")
print(f" OSNMA Anexo A     : {'PASSED' if snapshot['osnma']['annex_a_selftest_passed'] else 'FAIL'} (KROOT={snapshot['osnma']['latest_kroot_hex']})")
print('--------------------------------------------------------------------------------')
print(' Línea oficial .sol (33 columnas EUSPA HAS-UA):')
print(snapshot['has_ua']['sol_line_33col'])